# Project 02 — Portfolio-Grade MarianMT Fine-Tuning and Evaluation

This notebook upgrades the English↔Hindi neural machine translation project from a pretrained-model demonstration into a reproducible ML experiment.

It will:

1. verify the NVIDIA GPU and software environment;
2. load deterministic IIT Bombay English–Hindi train, validation, and test subsets;
3. evaluate the original pretrained MarianMT models;
4. fine-tune both translation directions;
5. evaluate the fine-tuned models on the same held-out test pairs;
6. calculate SacreBLEU, chrF, chrF++, TER, latency, throughput, preservation diagnostics, and bootstrap confidence intervals;
7. create a pretrained-versus-fine-tuned comparison;
8. generate a 30-example manual error-analysis worksheet;
9. update the project JSON, CSV, PNG, and Static Space metric files automatically.

**No result is fabricated. All committed metrics must come from running this notebook.**

## Before running

Open a terminal in `02-neural-machine-translation-transformer` and activate your preferred virtual environment.

Verify that your existing PyTorch installation detects the RTX GPU:

```cmd
python -c "import torch; print(torch.__version__); print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"
```

Then install the evaluation-only dependencies:

```cmd
pip install -r requirements-evaluation.txt
```

`requirements-evaluation.txt` intentionally does not reinstall PyTorch, because the correct CUDA build depends on your local NVIDIA environment.

## Evaluation profiles

| Profile | Training pairs | Test pairs | Epochs | Purpose |
|---|---:|---:|---:|---|
| `quick` | 5,000 | 250 | 1 | Verify that the complete workflow runs |
| `portfolio` | 50,000 | 1,000 | 2 | Recommended recruiter-facing experiment |
| `full` | 100,000 | 2,507 | 3 | Strongest experiment; longest runtime |

Start with `quick`. After it succeeds, delete or archive the quick outputs and rerun with `portfolio`.

In [1]:
from pathlib import Path
import json
import sys

CURRENT = Path.cwd().resolve()
PROJECT_ROOT = CURRENT.parent if CURRENT.name == "notebooks" else CURRENT
if not (PROJECT_ROOT / "configs" / "portfolio_evaluation.yaml").exists():
    raise FileNotFoundError(
        "Open this notebook from the Project 02 folder or its notebooks folder."
    )
sys.path.insert(0, str(PROJECT_ROOT))

from src.portfolio_evaluation import (
    collect_environment,
    compare_systems,
    create_manual_review_candidates,
    create_plots,
    detect_hardware,
    evaluate_system,
    fine_tune_both_directions,
    fine_tuned_model_refs,
    load_and_prepare_dataset,
    load_config,
    load_prepared_dataset,
    save_comparison_artifacts,
    save_prepared_dataset,
    set_reproducibility,
    summarize_manual_review,
    sync_portfolio_outputs,
)

print(PROJECT_ROOT)

C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\02-neural-machine-translation-transformer


## Step 1 — Select the run profile

Change `PROFILE` only after the quick run succeeds. The recommended final portfolio run is `portfolio`.

In [2]:
PROFILE = "portfolio"  # Change to "portfolio" for the final recruiter-facing run.

config = load_config(PROJECT_ROOT, profile=PROFILE)
set_reproducibility(int(config["seed"]))

print(json.dumps({
    "active_profile": config["active_profile"],
    "profile_settings": config["profile"],
    "dataset": config["dataset"]["id"],
    "models": config["models"],
}, indent=2))

{
  "active_profile": "portfolio",
  "profile_settings": {
    "train_pairs": 50000,
    "validation_pairs": 520,
    "test_pairs": 1000,
    "epochs": 2,
    "bootstrap_samples": 500,
    "manual_review_examples": 30
  },
  "dataset": "cfilt/iitb-english-hindi",
  "models": {
    "en_hi": "Helsinki-NLP/opus-mt-en-hi",
    "hi_en": "Helsinki-NLP/opus-mt-hi-en"
  }
}


## Step 2 — Validate the RTX GPU and save the environment

The notebook automatically chooses conservative batch sizes from available VRAM. This information is saved with the results so latency and training claims remain reproducible.

In [3]:
hardware = detect_hardware()
environment = collect_environment(hardware)

print(json.dumps({
    "hardware": hardware.__dict__,
    "environment": environment,
}, indent=2))

if hardware.device != "cuda":
    raise RuntimeError(
        "CUDA is not available. Stop here and repair the GPU-enabled PyTorch installation before fine-tuning."
    )

{
  "hardware": {
    "device": "cuda",
    "gpu_name": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gb": 31.84,
    "bf16_supported": true,
    "recommended_train_batch_size": 12,
    "recommended_eval_batch_size": 16,
    "gradient_accumulation_steps": 2
  },
  "environment": {
    "python": "3.13.14",
    "platform": "Windows-11-10.0.26200-SP0",
    "processor": "Intel64 Family 6 Model 198 Stepping 2, GenuineIntel",
    "device": "cuda",
    "gpu_name": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gb": 31.84,
    "bf16_supported": true,
    "torch": "2.13.0+cu132",
    "transformers": "5.14.1",
    "datasets": "4.8.5",
    "sacrebleu": "2.6.0",
    "pandas": "3.0.5",
    "numpy": "2.5.1"
  }
}


## Step 3 — Download and prepare the official IIT Bombay dataset

The workflow uses the official Hugging Face dataset repository and deterministic sampling with seed 42. It saves the exact train, validation, and test CSV files used for the experiment under `outputs/portfolio_evaluation/datasets/`.

In [4]:
frames = load_and_prepare_dataset(config)
dataset_files = save_prepared_dataset(frames, config, environment)

print(dataset_files)
for split_name, frame in frames.items():
    print(split_name, frame.shape)
    display(frame.head(3))

{'train': 'outputs\\portfolio_evaluation\\datasets\\train_pairs.csv', 'validation': 'outputs\\portfolio_evaluation\\datasets\\validation_pairs.csv', 'test': 'outputs\\portfolio_evaluation\\datasets\\test_pairs.csv', 'manifest': 'outputs\\portfolio_evaluation\\dataset_manifest.json'}
train (50000, 2)


,english,hindi
0,on the intuition.,अंतर्ज्ञान पर।
1,Ceiba pentandra,कण्टकारी
2,Have you not regarded how your Lord dealt with...,क्या तुमने देखा नहीं कि तुम्हारे आद के साथ क्य...


validation (520, 2)


,english,hindi
0,Students of the Dattatreya city Municipal corp...,महानगर पालिका अंतर्गत दत्तात्रय नगर माध्यमिक स...
1,With encouragement from Principal Sandhya Medp...,प्रधानाध्यापक संध्या मेडपल्लीवार के प्रोत्साहि...
2,"Rajesh Gavre, the President of the MNPA teache...",मनपा शिक्षक संघ के अध्यक्ष राजेश गवरे ने स्कूल...


test (1000, 2)


,english,hindi
0,They never believed he died the way the sherif...,उन्हें शेरिफ के बताये गये तरीके से मृत्यु पर ब...
1,People crowded the jewellery shops in Simla.,शिमला में आभूषणों की दुकानों में लोंगों का हुज...
2,Many on Wall Street view the addition of bagga...,वॉल स्ट्रीट पर बैगेज की फीस एक संकेत के रूप मे...


### Dataset sanity checks

In [5]:
dataset_summary = {
    split: {
        "rows": len(frame),
        "english_avg_characters": round(frame["english"].str.len().mean(), 2),
        "hindi_avg_characters": round(frame["hindi"].str.len().mean(), 2),
        "duplicate_pairs": int(frame.duplicated(["english", "hindi"]).sum()),
        "missing_english": int(frame["english"].isna().sum()),
        "missing_hindi": int(frame["hindi"].isna().sum()),
    }
    for split, frame in frames.items()
}
print(json.dumps(dataset_summary, indent=2, ensure_ascii=False))

{
  "train": {
    "rows": 50000,
    "english_avg_characters": 73.32,
    "hindi_avg_characters": 71.19,
    "duplicate_pairs": 0,
    "missing_english": 0,
    "missing_hindi": 0
  },
  "validation": {
    "rows": 520,
    "english_avg_characters": 105.02,
    "hindi_avg_characters": 90.77,
    "duplicate_pairs": 0,
    "missing_english": 0,
    "missing_hindi": 0
  },
  "test": {
    "rows": 1000,
    "english_avg_characters": 115.79,
    "hindi_avg_characters": 115.58,
    "duplicate_pairs": 0,
    "missing_english": 0,
    "missing_hindi": 0
  }
}


## Step 4 — Evaluate the original pretrained MarianMT models

This is the required baseline. Both directions are evaluated on the exact same held-out test pairs that will later be used for the fine-tuned models.

Generated files include:

- direction-wise prediction CSV files;
- SacreBLEU and full metric signatures;
- chrF, chrF++, and TER;
- average, median, P95, minimum, and maximum latency;
- throughput and peak GPU memory;
- bootstrap 95% confidence intervals;
- number-preservation and script diagnostics.

In [6]:
pretrained_metrics = evaluate_system(
    frames["test"],
    system_name="pretrained",
    model_refs=config["models"],
    hardware=hardware,
    config=config,
)
print(json.dumps(pretrained_metrics, indent=2, ensure_ascii=False))

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=160) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=160) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=160) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=160) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=160) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=160) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=160) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=160) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

{
  "en_hi": {
    "status": "evaluated",
    "system": "pretrained",
    "direction": "en_hi",
    "model_ref": "Helsinki-NLP/opus-mt-en-hi",
    "examples": 1000,
    "metrics": {
      "sacrebleu": 9.6561,
      "sacrebleu_signature": "nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0",
      "chrf": 32.1131,
      "chrf_signature": "nrefs:1|case:mixed|eff:yes|nc:6|nw:0|space:no|version:2.6.0",
      "chrf_plus_plus": 30.5585,
      "chrf_plus_plus_signature": "nrefs:1|case:mixed|eff:yes|nc:6|nw:2|space:no|version:2.6.0",
      "ter": 81.9286,
      "ter_signature": "nrefs:1|case:lc|tok:tercom|norm:yes|punct:yes|asian:no|version:2.6.0"
    },
    "runtime": {
      "system": "pretrained",
      "direction": "en_hi",
      "model_ref": "Helsinki-NLP/opus-mt-en-hi",
      "device": "cuda",
      "batch_size": 16,
      "sentences": 1000,
      "total_inference_seconds": 23.225199,
      "average_latency_seconds": 0.023225,
      "median_latency_seconds": 0.021871,
      "p95_

### Compact baseline table

In [7]:
import pandas as pd

baseline_table = pd.DataFrame([
    {
        "system": "pretrained",
        "direction": direction,
        "SacreBLEU": result["metrics"]["sacrebleu"],
        "chrF": result["metrics"]["chrf"],
        "chrF++": result["metrics"]["chrf_plus_plus"],
        "TER": result["metrics"]["ter"],
        "Avg latency (s)": result["runtime"]["average_latency_seconds"],
        "P95 latency (s)": result["runtime"]["p95_latency_seconds"],
        "Sentences/sec": result["runtime"]["sentences_per_second"],
        "Number preservation": result["diagnostics"]["number_preservation_rate"],
    }
    for direction, result in pretrained_metrics.items()
])
display(baseline_table)

,system,direction,SacreBLEU,chrF,chrF++,TER,Avg latency (s),P95 latency (s),Sentences/sec,Number preservation
0,pretrained,en_hi,9.6561,32.1131,30.5585,81.9286,0.023225,0.034148,43.0567,0.921
1,pretrained,hi_en,13.3014,40.6827,38.5463,70.8942,0.020338,0.027700,49.1685,0.929


## Step 5 — Fine-tune MarianMT in both directions

This stage trains two models sequentially:

- `models/fine_tuned_en_hi`
- `models/fine_tuned_hi_en`

Model weights remain excluded from Git because they are large. Training summaries and histories are saved under `outputs/portfolio_evaluation/training/` and should be committed.

The notebook uses mixed precision, gradient checkpointing, early stopping, best-checkpoint restoration, and GPU-aware batch sizing.

In [8]:
RUN_FINE_TUNING = True

if RUN_FINE_TUNING:
    training_summaries = fine_tune_both_directions(frames, hardware, config)
    print(json.dumps(training_summaries, indent=2, ensure_ascii=False))
else:
    print("Fine-tuning skipped. Set RUN_FINE_TUNING=True to create the portfolio-grade models.")

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/520 [00:00<?, ? examples/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Epoch,Training Loss,Validation Loss,Sacrebleu,Chrf
1,8.147271,4.753174,9.145700,34.062900
2,7.391832,4.704893,9.721500,35.005200


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.encoder.embed_positions.weight', 'model.decoder.embed_tokens.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "en_hi": {
    "status": "trained",
    "direction": "en_hi",
    "base_model": "Helsinki-NLP/opus-mt-en-hi",
    "output_dir": "models\\fine_tuned_en_hi",
    "train_examples": 50000,
    "validation_examples": 520,
    "epochs_requested": 2,
    "training_seconds": 629.135,
    "best_checkpoint": "C:\\Users\\atripathi\\OneDrive - Veralto\\Desktop\\AI Codes\\GIT Projects\\transformer-projects\\02-neural-machine-translation-transformer\\outputs\\portfolio_evaluation\\training\\en_hi\\checkpoint-4168",
    "best_metric": 35.0052,
    "train_metrics": {
      "train_runtime": 628.7333,
      "train_samples_per_second": 159.05,
      "train_steps_per_second": 6.629,
      "total_flos": 1389513046228992.0,
      "train_loss": 7.9441355377607294,
      "epoch": 2.0
    },
    "hardware": {
      "device": "cuda",
      "gpu_name": "NVIDIA GeForce RTX 5090",
      "gpu_memory_gb": 31.84,
      "bf16_supported": true,
      "recommended_train_batch_size": 12,
      "recommended_eval_batch

## Step 6 — Evaluate the fine-tuned models

Do not compare models on different test subsets. This cell reuses the saved held-out pairs from Step 3.

In [9]:
# Reload the exact saved splits in case the notebook was restarted after training.
frames = load_prepared_dataset(config)

fine_tuned_metrics = evaluate_system(
    frames["test"],
    system_name="fine_tuned",
    model_refs=fine_tuned_model_refs(config),
    hardware=hardware,
    config=config,
)
print(json.dumps(fine_tuned_metrics, indent=2, ensure_ascii=False))

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=160) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=160) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=160) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=160) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=160) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=160) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=160) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=160) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

{
  "en_hi": {
    "status": "evaluated",
    "system": "fine_tuned",
    "direction": "en_hi",
    "model_ref": "C:\\Users\\atripathi\\OneDrive - Veralto\\Desktop\\AI Codes\\GIT Projects\\transformer-projects\\02-neural-machine-translation-transformer\\models\\fine_tuned_en_hi",
    "examples": 1000,
    "metrics": {
      "sacrebleu": 12.5665,
      "sacrebleu_signature": "nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0",
      "chrf": 36.7752,
      "chrf_signature": "nrefs:1|case:mixed|eff:yes|nc:6|nw:0|space:no|version:2.6.0",
      "chrf_plus_plus": 35.1802,
      "chrf_plus_plus_signature": "nrefs:1|case:mixed|eff:yes|nc:6|nw:2|space:no|version:2.6.0",
      "ter": 75.7373,
      "ter_signature": "nrefs:1|case:lc|tok:tercom|norm:yes|punct:yes|asian:no|version:2.6.0"
    },
    "runtime": {
      "system": "fine_tuned",
      "direction": "en_hi",
      "model_ref": "C:\\Users\\atripathi\\OneDrive - Veralto\\Desktop\\AI Codes\\GIT Projects\\transformer-projects\\02-neu

## Step 7 — Compare pretrained and fine-tuned systems

The comparison uses paired bootstrap resampling because both systems translated the same source sentences. This provides a more credible result than reporting only raw score differences.

In [10]:
comparison, comparison_summary = compare_systems(config)
comparison_files = save_comparison_artifacts(comparison, comparison_summary, config)
plot_files = create_plots(comparison, config)

display(comparison)
print(json.dumps(comparison_summary["paired_bootstrap_significance"], indent=2))
print(comparison_files)
print(plot_files)

,system,direction,model_ref,examples,sacrebleu,chrf,chrf_plus_plus,ter,average_latency_seconds,p95_latency_seconds,sentences_per_second,number_preservation_rate
0,pretrained,en_hi,Helsinki-NLP/opus-mt-en-hi,1000,9.6561,32.1131,30.5585,81.9286,0.023225,0.034148,43.0567,0.921
1,fine_tuned,en_hi,C:\Users\atripathi\OneDrive - Veralto\Desktop\...,1000,12.5665,36.7752,35.1802,75.7373,0.023610,0.034823,42.3548,0.928
2,pretrained,hi_en,Helsinki-NLP/opus-mt-hi-en,1000,13.3014,40.6827,38.5463,70.8942,0.020338,0.027700,49.1685,0.929
3,fine_tuned,hi_en,C:\Users\atripathi\OneDrive - Veralto\Desktop\...,1000,14.0044,41.5662,39.6603,71.7204,0.022530,0.033359,44.3854,0.928


{
  "en_hi": {
    "method": "paired bootstrap resampling",
    "samples": 500,
    "seed": 42,
    "examples": 1000,
    "sacrebleu": {
      "mean_delta": 2.8866,
      "lower_95": 2.297,
      "upper_95": 3.5247,
      "probability_candidate_better": 1.0
    },
    "chrf": {
      "mean_delta": 4.6874,
      "lower_95": 4.0918,
      "upper_95": 5.3301,
      "probability_candidate_better": 1.0
    }
  },
  "hi_en": {
    "method": "paired bootstrap resampling",
    "samples": 500,
    "seed": 42,
    "examples": 1000,
    "sacrebleu": {
      "mean_delta": 0.6901,
      "lower_95": 0.0373,
      "upper_95": 1.339,
      "probability_candidate_better": 0.98
    },
    "chrf": {
      "mean_delta": 0.8946,
      "lower_95": 0.3663,
      "upper_95": 1.3855,
      "probability_candidate_better": 1.0
    }
  }
}
{'comparison': 'outputs\\portfolio_evaluation\\model_comparison.csv', 'summary': 'outputs\\portfolio_evaluation\\comparison_summary.json'}
['outputs\\portfolio_evaluation\\plot

### Improvement table

In [11]:
improvements = []
for direction in ["en_hi", "hi_en"]:
    subset = comparison[comparison["direction"] == direction].set_index("system")
    improvements.append({
        "direction": direction,
        "SacreBLEU improvement": round(subset.loc["fine_tuned", "sacrebleu"] - subset.loc["pretrained", "sacrebleu"], 4),
        "chrF improvement": round(subset.loc["fine_tuned", "chrf"] - subset.loc["pretrained", "chrf"], 4),
        "TER reduction": round(subset.loc["pretrained", "ter"] - subset.loc["fine_tuned", "ter"], 4),
        "Latency change (s)": round(subset.loc["fine_tuned", "average_latency_seconds"] - subset.loc["pretrained", "average_latency_seconds"], 6),
    })
display(pd.DataFrame(improvements))

,direction,SacreBLEU improvement,chrF improvement,TER reduction,Latency change (s)
0,en_hi,2.9104,4.6621,6.1913,0.000385
1,hi_en,0.7030,0.8835,-0.8262,0.002192


## Step 8 — Generate the manual error-analysis worksheet

The notebook selects the weakest 30 fine-tuned translations using sentence-level metrics and heuristics. The output is:

`outputs/portfolio_evaluation/manual_error_analysis_candidates.csv`

Open it in Excel and fill these four columns:

- `human_error_category`: good_translation, word_order, missing_information, named_entity, number_or_date, gender_or_tense, over_literal, under_translation, over_translation, mixed_language, technical_terminology, other
- `human_severity`: low, medium, high
- `human_translation_quality`: good, acceptable, weak
- `human_notes`: short explanation

Do not alter the source, reference, prediction, or metric columns.

In [12]:
import importlib
import src.portfolio_evaluation as portfolio_evaluation

portfolio_evaluation = importlib.reload(portfolio_evaluation)
create_manual_review_candidates = (
    portfolio_evaluation.create_manual_review_candidates
)

print("Corrected evaluation module reloaded successfully.")

Corrected evaluation module reloaded successfully.


In [13]:
manual_candidates = create_manual_review_candidates(config)
manual_path = PROJECT_ROOT / config["outputs"]["root"] / "manual_error_analysis_candidates.csv"
manual_candidates.to_csv(manual_path, index=False, encoding="utf-8-sig")
print(f"Created: {manual_path}")
display(manual_candidates.head(10))

Created: C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\02-neural-machine-translation-transformer\outputs\portfolio_evaluation\manual_error_analysis_candidates.csv


,direction,source_text,reference_translation,predicted_translation,sentence_bleu,sentence_chrf,sentence_ter,heuristic_error_category,human_error_category,human_severity,human_translation_quality,human_notes
1193,hi_en,राष्ट्रपति अर्मांडो जीबुझां ने अस्थिरता के बार...,President Armando Guebuza has sought to play d...,PRESIDENT THE PRESIDENT OF THE PRESIDENTS HAVE...,2.6280,0.9718,100.0000,low_metric_review,,,,
341,en_hi,"Ejaz Jaan, the legislator of the Poonch Assemb...",पुंछ विधानसभा क्षेत्र के विधायक एजाज जान ने गु...,"पंडी सम्मेलन समिति के सहकारी, एहाज जे. जे. पी....",1.1231,7.5421,200.0000,low_metric_review,,,,
490,en_hi,Where does this leave Scotland?,इन सबसे पश्चात स्कॉटलैंड का क्या होगा?,यह बाघ को कहां छोड़ देता है?,5.5224,8.9081,87.5000,low_metric_review,,,,
1665,hi_en,फ्री लिमो तथा रे नमो दोनों ही जोर देते हैं कि ...,Both Frelimo and Renamo insist they want to av...,Unable to escape war.,7.1213,9.3938,72.7273,possible_under_translation,,,,
228,en_hi,Wallis Simpson may have been intersex.,हो सकता है कि वैलिस सिम्पसन इंटरसेक्स रहे हों।,विपणन शायद एक-दूसरे के बीच रहा हो।,0.0000,9.7884,100.0000,low_metric_review,,,,
716,en_hi,Presenting CAT's trendy footwear and clothing ...,कैट की ट्रैडी फुटवियर और परिधानों की सिलेक्शन पेश,सी. सी. ए. के उद्योग और कपड़े संग्रह प्रस्तुत ...,3.0891,10.5379,133.3333,low_metric_review,,,,
440,en_hi,Only wealth gained with self-respect is long l...,वही धन-संपदा टिकाऊ होती है जिसके साथ सम्मान जु...,केवल अपना माल ही काफी स्थायी है।,0.0000,10.5891,100.0000,low_metric_review,,,,
1578,hi_en,अब इन पर 12.50 फीसद वैट व 1.50 फीसद अतिरिक्त क...,Now a 12.5% VAT plus a 1.5% additional tax on ...,Now they have 1250 fees and 1.50 fees.,2.3309,11.0354,88.2353,possible_under_translation,,,,
1710,hi_en,धन को हम पवित्र उद्देश्य से कमाएं और पवित्र उद...,Wealth should be earned with a sacred objectiv...,Let us use money for the holy purpose and for ...,2.7258,11.1769,88.2353,low_metric_review,,,,
999,en_hi,This created a cold front in the valley.,इसके साथ ही पूरी वादी में शीतलहर शुरू हो गई है।,यह घाटी में एक शीत स्तर बनाया गया।,3.7955,11.6300,90.9091,low_metric_review,,,,


## Step 9 — Summarize the completed human review

Run this cell after saving the manually edited CSV. It is acceptable for the first run to report `awaiting_human_review`.

In [18]:
review = pd.read_csv(manual_path)
manual_summary = summarize_manual_review(review)
manual_summary_path = PROJECT_ROOT / config["outputs"]["root"] / "manual_error_analysis_summary.json"
manual_summary_path.write_text(
    json.dumps(manual_summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(json.dumps(manual_summary, ensure_ascii=False, indent=2))

{
  "status": "completed",
  "reviewed_examples": 30,
  "error_category_counts": {
    "other": 9,
    "named_entity": 6,
    "missing_information": 4,
    "number_or_date": 3,
    "technical_terminology": 3,
    "word_order": 2,
    "good_translation": 2,
    "under_translation": 1
  },
  "severity_counts": {
    "high": 19,
    "medium": 8,
    "low": 3
  },
  "quality_counts": {
    "weak": 21,
    "acceptable": 7,
    "good": 2
  },
  "mean_sentence_chrf": 12.0736,
  "notes_completed": 30
}


## Step 10 — Populate all portfolio and Static Space result files

This final cell updates:

- `outputs/sacrebleu_scores.json`
- `outputs/chrf_scores.json`
- `outputs/translation_latency_results.json`
- `outputs/model_metrics.json`
- `outputs/model_comparison.csv`
- `web/data/evaluation-results.json`

The Static Space will then display the verified fine-tuned SacreBLEU and chrF values. The current browser model is still the quantized pretrained ONNX model; converting the fine-tuned checkpoints to ONNX is a separate deployment step and must be completed before claiming that the browser demo uses the fine-tuned models.

In [19]:
comparison, comparison_summary = compare_systems(config)
review = pd.read_csv(manual_path) if manual_path.exists() else pd.DataFrame()
manual_summary = (
    summarize_manual_review(review)
    if not review.empty
    else {"status": "not_created", "reviewed_examples": 0}
)

synced_files = sync_portfolio_outputs(
    comparison,
    comparison_summary,
    manual_summary,
    config,
)
print(json.dumps(synced_files, indent=2))

{
  "web_metrics": "web\\data\\evaluation-results.json",
  "sacrebleu": "outputs/sacrebleu_scores.json",
  "chrf": "outputs/chrf_scores.json",
  "latency": "outputs/translation_latency_results.json",
  "model_metrics": "outputs/model_metrics.json",
  "comparison": "outputs/model_comparison.csv"
}


## Final portfolio checklist

Before pushing the final results:

- [ ] The final profile is `portfolio` or `full`, not `quick`.
- [ ] Both pretrained directions were evaluated.
- [ ] Both fine-tuned directions were evaluated on the same test set.
- [ ] SacreBLEU signatures are present in the direction metric JSON files.
- [ ] Bootstrap intervals and paired comparison results are present.
- [ ] At least 30 translations received human review.
- [ ] The root output JSON files no longer contain `null` metric placeholders.
- [ ] `web/data/evaluation-results.json` contains the verified values.
- [ ] Model checkpoints are not staged in Git.
- [ ] The README is updated with actual values only after this run.

### Files that should be committed

Commit the notebook, configuration, scripts, source modules, JSON/CSV/PNG evaluation artifacts, manual review, and Static Space metrics.

### Files that should not be committed

Do not commit `models/fine_tuned_en_hi/`, `models/fine_tuned_hi_en/`, dataset caches, virtual environments, or Hugging Face cache folders. The fine-tuned model weights should later be uploaded to dedicated Hugging Face Model repositories.